In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

from tarifalatam.config import GROUP_COL
from tarifalatam.data.cleaning import check_currencies, load_raw, parse_price

# 02 · EDA — qué hay realmente en los anuncios de TarifaLatam

Este notebook **reproduce** las cifras de `reports/01_eda.md` y termina con una tabla que compara cada cifra con el
informe. Toda la lógica reutilizable vive en `src/tarifalatam/`; aquí solo hay resúmenes de una línea.

**Datos personales:** `load_raw()` elimina las columnas del anfitrión (nombre, presentación, fotos, URLs) y el texto libre
(`name`, `description`, `neighborhood_overview`) **antes** de devolver la tabla. Ninguna celda imprime filas completas: solo
conteos y resúmenes.

**Preguntas guía:** (1) unidad de análisis · (2) objetivo · (3) suciedad · (4) qué debería ser estable · (5) fuga.

## 0. Cargar los datos

In [2]:
df = load_raw()  # siete archivos de data/raw/, sin datos personales ni texto libre
prices = parse_price(df)["price_local"]  # misma regla que usa la limpieza: "$1,234.00" -> 1234.0
print(f"{len(df):,} filas x {df.shape[1]} columnas (incluye 'city', agregada al cargar)")
df.groupby("city").size().rename("anuncios").to_frame().T

194,628 filas x 78 columnas (incluye 'city', agregada al cargar)


city,belize,bogota,buenos-aires,mexico-city,rio-de-janeiro,santiago,sao-paulo
anuncios,3671,19505,29753,30468,48247,18904,44080


## 1. ¿Cuál es la unidad de análisis?

*Pregunta:* ¿una fila es un anuncio? ¿alguna entidad aparece en varias filas?

In [3]:
per_host = df.groupby(GROUP_COL).size()
summary_unit = pd.Series(
    {
        "id repetidos": df["id"].duplicated().sum(),
        "anfitriones": per_host.size,
        "anuncios por anfitrión (promedio)": round(per_host.mean(), 1),
        "mediana": per_host.median(),
        "percentil 99": per_host.quantile(0.99),
        "máximo": per_host.max(),
        "% anfitriones con 1 anuncio": round((per_host == 1).mean() * 100, 1),
        "% anuncios de anfitriones con >= 10": round(per_host[per_host >= 10].sum() / len(df) * 100, 1),
        "anfitriones en más de una ciudad": (df.groupby(GROUP_COL)["city"].nunique() > 1).sum(),
    }
)
summary_unit.to_frame("valor")

,valor
id repetidos,0.0
anfitriones,89964.0
anuncios por anfitrión (promedio),2.2
mediana,1.0
percentil 99,19.0
máximo,699.0
% anfitriones con 1 anuncio,73.0
% anuncios de anfitriones con >= 10,31.0
anfitriones en más de una ciudad,258.0


**Respuesta:** una fila es **un anuncio** (`id` no se repite) en la fecha de corte de su ciudad. El anfitrión
(`host_id`) se repite: el 31 % de los anuncios es de anfitriones con 10 o más, y 258 anfitriones están en varias ciudades.

**Unidad por la que se parten los datos:** `host_id` (ver `src/tarifalatam/data/split.py`).

## 2. ¿Cómo está el objetivo?

*Pregunta:* ¿cuántos anuncios no tienen precio? ¿qué forma tiene la distribución? ¿cuál es el precio típico?

In [4]:
by_city = pd.DataFrame(
    {
        "anuncios": df.groupby("city").size(),
        "sin precio": prices.isna().groupby(df["city"]).sum(),
        "% sin precio": (prices.isna().groupby(df["city"]).mean() * 100).round(2),
        "asimetría (escala original)": prices.groupby(df["city"]).skew().round(1),
        "asimetría (logaritmo)": np.log(prices).groupby(df["city"]).skew().round(2),
    }
)
print(f"anuncios sin precio: {prices.isna().sum():,} ({prices.isna().mean():.1%})")
by_city

anuncios sin precio: 11,642 (6.0%)


,anuncios,sin precio,% sin precio,asimetría (escala original),asimetría (logaritmo)
city,,,,,
belize,3671,167,4.55,23.4,0.54
bogota,19505,711,3.65,55.6,0.97
buenos-aires,29753,2119,7.12,29.7,2.09
mexico-city,30468,2108,6.92,97.0,0.65
rio-de-janeiro,48247,4100,8.50,90.3,1.01
santiago,18904,898,4.75,62.4,1.54
sao-paulo,44080,1539,3.49,160.3,0.57


In [5]:
# Mediana del precio por noche, en moneda local (aún no está en USD)
medians = prices.groupby([df["city"], df["room_type"]]).median().unstack().round(0)
shares = (df["room_type"].value_counts(normalize=True) * 100).round(1)
print("% de anuncios por tipo de habitación:", shares.to_dict())
medians

% de anuncios por tipo de habitación: {'Entire home/apt': 80.6, 'Private room': 18.4, 'Shared room': 0.8, 'Hotel room': 0.3}


room_type,Entire home/apt,Hotel room,Private room,Shared room
city,,,,
belize,403.0,250.0,260.0,38.0
bogota,181250.0,195000.0,90000.0,40000.0
buenos-aires,110075.0,253616.0,52836.0,30577.0
mexico-city,1828.0,2300.0,849.0,400.0
rio-de-janeiro,491.0,413.0,290.0,120.0
santiago,60000.0,105784.0,32716.0,22000.0
sao-paulo,353.0,594.0,181.0,90.0


**Hallazgo:** el 6,0 % de los anuncios no tiene precio (entre 3,5 % y 8,5 % según la ciudad). El precio es muy
asimétrico; en logaritmo la asimetría baja a valores entre 0,5 y 2,1, por eso se modela `log(price_usd)`.

**Línea base:** mediana del precio por ciudad × `room_type`. Su MAE en USD se calcula en el notebook 03, con el
entrenamiento, cuando estén las tasas de cambio.

## 3. ¿Qué tan sucios están los datos y de qué manera?

### 3.1 Columnas vacías, números como texto y centinelas

In [6]:
empty_by_city = {city: {c for c in part.columns if part[c].isna().all()} for city, part in df.groupby("city")}
empty_everywhere = sorted(set.intersection(*empty_by_city.values()))
print(f"{len(empty_everywhere)} columnas 100 % vacías en las siete ciudades:", empty_everywhere)
print("(neighborhood_overview también estaba vacía, pero load_raw ya la quitó como texto libre)")
print("vacías solo en algunas ciudades:", sorted(set.union(*empty_by_city.values()) - set(empty_everywhere)))

baths = df["bathrooms_text"].dropna()
no_number = baths[~baths.str.contains(r"\d", regex=True)]
print(f"\nbathrooms_text vacío: {df['bathrooms_text'].isna().mean():.1%} | sin número: {len(no_number):,}",
      no_number.value_counts().to_dict())
print(f"baños compartidos: {baths.str.contains('shared', case=False).mean():.1%}")

print("\nmaximum_nights = 1.125 (por defecto):", (df["maximum_nights"] == 1125).sum(),
      "| > 1.125:", (df["maximum_nights"] > 1125).sum(),
      "| = 2.147.483.647:", (df["maximum_nights"] == 2147483647).sum())

11 columnas 100 % vacías en las siete ciudades: ['bathrooms', 'calendar_updated', 'host_acceptance_rate', 'host_neighbourhood', 'host_response_rate', 'host_response_time', 'host_since', 'host_total_listings_count', 'host_verifications', 'instant_bookable', 'neighbourhood']
(neighborhood_overview también estaba vacía, pero load_raw ya la quitó como texto libre)
vacías solo en algunas ciudades: ['license', 'neighbourhood_group_cleansed']

bathrooms_text vacío: 0.1% | sin número: 1,018 {'Shared half-bath': 400, 'Private half-bath': 368, 'Half-bath': 241, 'No bathroom': 8, 'Shared half bathroom': 1}
baños compartidos: 8.0%

maximum_nights = 1.125 (por defecto): 60322 | > 1.125: 50 | = 2.147.483.647: 21


### 3.2 Valores imposibles o fuera del caso

In [7]:
monthly = df["minimum_nights"] >= 30
bogota = df["city"] == "bogota"
print(f"minimum_nights >= 30: {monthly.sum():,} ({monthly.mean():.1%}) | en Bogotá: {monthly[bogota].mean():.1%}")
print("Bogotá, mediana del precio por noche (COP):",
      prices[bogota].groupby(monthly[bogota]).median().rename({False: "< 30 noches", True: ">= 30 noches"}).to_dict())

print(f"\nbedrooms = 0: {(df['bedrooms'] == 0).sum():,} | > 10: {(df['bedrooms'] > 10).sum():,}",
      "| > 10 y > capacidad:", ((df["bedrooms"] > 10) & (df["bedrooms"] > df["accommodates"])).sum())
print("bedrooms > 10 por tipo:", df.loc[df["bedrooms"] > 10, "room_type"].value_counts().to_dict())

ratio = prices / prices.groupby(df["city"]).transform("median")
print(f"\nprecio > 20 x la mediana de su ciudad: {(ratio > 20).sum():,} | > 10 x: {(ratio > 10).sum():,}",
      f"| < 0,1 x: {(ratio < 0.1).sum():,}")
print("> 20 x por ciudad:", df.loc[ratio > 20, "city"].value_counts().to_dict())
print("máximo por ciudad (moneda local):", prices.groupby(df["city"]).max().round(0).to_dict())

minimum_nights >= 30: 6,103 (3.1%) | en Bogotá: 16.9%
Bogotá, mediana del precio por noche (COP): {'< 30 noches': 170000.0, '>= 30 noches': 94102.095}

bedrooms = 0: 10,502 | > 10: 220 | > 10 y > capacidad: 124
bedrooms > 10 por tipo: {'Private room': 129, 'Entire home/apt': 81, 'Hotel room': 9, 'Shared room': 1}

precio > 20 x la mediana de su ciudad: 692 | > 10 x: 1,898 | < 0,1 x: 61
> 20 x por ciudad: {'rio-de-janeiro': 355, 'buenos-aires': 104, 'mexico-city': 76, 'sao-paulo': 60, 'bogota': 44, 'santiago': 38, 'belize': 15}
máximo por ciudad (moneda local): {'belize': 55410.0, 'bogota': 258514678.0, 'buenos-aires': 160563760.0, 'mexico-city': 1826199.0, 'rio-de-janeiro': 587008.0, 'santiago': 97573964.0, 'sao-paulo': 500210.0}


### 3.3 Vacíos que significan algo

In [8]:
no_reviews = df["number_of_reviews"] == 0
print(f"sin reseñas: {no_reviews.mean():.1%}",
      f"| nota vacía entre los sin reseñas: {df.loc[no_reviews, 'review_scores_rating'].isna().mean():.0%}",
      f"| nota vacía con reseñas: {(~no_reviews & df['review_scores_rating'].isna()).sum()}")

no_price = prices.isna()
closed = df["availability_365"] == 0
print(f"calendario cerrado (availability_365 = 0): sin precio {closed[no_price].mean():.1%}",
      f"| con precio {closed[~no_price].mean():.1%}")

previous = df["source"] == "previous scrape"
print(f"source = 'previous scrape': {previous.sum():,} ({previous.mean():.1%})",
      f"| de ellos sin precio: {no_price[previous].mean():.1%} | con precio: {(previous & ~no_price).sum():,}")

sin reseñas: 15.5% | nota vacía entre los sin reseñas: 100% | nota vacía con reseñas: 0
calendario cerrado (availability_365 = 0): sin precio 64.0% | con precio 0.2%
source = 'previous scrape': 11,701 (6.0%) | de ellos sin precio: 71.9% | con precio: 3,293


### 3.4 Duplicados

In [9]:
twins = ["host_id", "latitude", "longitude", "room_type", "accommodates", "bedrooms"]
print("filas idénticas salvo el id:", df.drop(columns=["id"]).duplicated().sum())
print("mismo anfitrión, coordenadas, tipo, capacidad, habitaciones y precio:",
      df.assign(price_local=prices).duplicated(subset=twins + ["price_local"]).sum())

filas idénticas salvo el id: 159
mismo anfitrión, coordenadas, tipo, capacidad, habitaciones y precio: 3076


**Hallazgos de calidad:** 12 columnas vacías en todas las ciudades; baños escritos como texto (1.010 "half-bath" sin
número); `maximum_nights` con centinelas; arriendos mensuales concentrados en Bogotá; precios imposibles; los anuncios sin
precio son inactivos y los sin reseñas no tienen nota. Cada uno es una decisión de `reports/01_eda.md`.

## 4. ¿Qué variables deberían ser estables y no lo son?

*Pregunta:* lo que describe al anfitrión no debería cambiar entre sus anuncios.

In [10]:
stable = ["host_is_superhost", "host_listings_count", "hosts_time_as_host_years", "host_identity_verified"]
multi = df[df[GROUP_COL].isin(per_host[per_host > 1].index)]
(multi.groupby(GROUP_COL)[stable].nunique().gt(1).mean() * 100).round(2).rename("% de anfitriones con más de un valor").to_frame()

,% de anfitriones con más de un valor
host_is_superhost,0.00
host_listings_count,0.18
hosts_time_as_host_years,0.02
host_identity_verified,0.00


**Hallazgo:** casi nada cambia (como mucho el 0,18 % de los anfitriones en `host_listings_count`). No hay ruido que
reparar por anfitrión.

## 5. ¿Hay algo que suene a fuga de información?

*Pregunta:* ¿alguna columna contiene el precio o solo se conoce después de fijarlo?

In [11]:
quote = pd.to_numeric(df["price_quote_price_per_night"], errors="coerce")
both = prices.notna() & quote.notna() & (quote > 0)
price_ratio = prices[both] / quote[both]
print(f"price / price_quote_price_per_night: mínimo {price_ratio.min():.4f}, máximo {price_ratio.max():.4f}",
      f"en {both.sum():,} anuncios")

revenue = pd.to_numeric(df["estimated_revenue_l365d"], errors="coerce")
expected_revenue = prices * df["estimated_occupancy_l365d"]
ok = revenue.notna() & (expected_revenue > 0)
print(f"estimated_revenue_l365d / (precio x ocupación): mediana {(revenue[ok] / expected_revenue[ok]).median():.3f}")

currency = df["price_quote_raw"].map(
    lambda raw: (json.loads(raw).get("quote") or {}).get("currency") if isinstance(raw, str) and raw.startswith("{") else None
)
check_currencies(df)  # falla si alguna moneda no coincide con config.CITY_CURRENCY
pd.crosstab(df["city"], currency)

price / price_quote_price_per_night: mínimo 1.0000, máximo 1.0000 en 182,983 anuncios
estimated_revenue_l365d / (precio x ocupación): mediana 1.000


price_quote_raw,ARS,BRL,CLP,COP,MXN,USD
city,,,,,,
belize,0,0,0,0,0,3504
bogota,0,0,0,19076,0,0
buenos-aires,27760,0,0,0,0,0
mexico-city,0,0,0,0,28383,0
rio-de-janeiro,0,44202,0,0,0,0
santiago,0,0,18037,0,0,0
sao-paulo,0,42652,0,0,0,0


In [12]:
# Buenos Aires: muchos precios repetidos son múltiplos exactos de 764,41 ARS
ba = prices[df["city"] == "buenos-aires"].value_counts().head(6)
pd.DataFrame({"anuncios": ba, "precio / 764,41": (ba.index / 764.41).round(2)})

,anuncios,"precio / 764,41"
price_local,,
99373.0,220,130.0
91729.0,219,120.0
84085.0,179,110.0
76441.0,146,100.0
61153.0,105,80.0
107018.0,96,140.0


**Columnas sospechosas de fuga:** `price_quote_*` (idéntica al precio) y `estimated_revenue_l365d` (precio ×
ocupación) se eliminan. Ocupación y disponibilidad son consecuencia del precio: se miden con y sin ellas en la etapa 4.
Las monedas coinciden con `config.CITY_CURRENCY` (Belice en USD).

## 6. ¿Coinciden las cifras con el informe?

Los valores esperados están copiados de `reports/01_eda.md`. Si una cifra no coincide, **se corrige el informe**, no el
código.

In [13]:
pct_no_price = (prices.isna().groupby(df["city"]).mean() * 100).round(2)
checks = [
    ("filas", 194_628, len(df)),
    ("anfitriones", 89_964, per_host.size),
    ("% anuncios de anfitriones con >= 10", 31.0, round(per_host[per_host >= 10].sum() / len(df) * 100, 1)),
    ("anuncios sin precio", 11_642, int(prices.isna().sum())),
    ("% sin precio Belice", 4.55, pct_no_price["belize"]),
    ("% sin precio Bogotá", 3.65, pct_no_price["bogota"]),
    ("% sin precio Buenos Aires", 7.12, pct_no_price["buenos-aires"]),
    ("% sin precio Ciudad de México", 6.92, pct_no_price["mexico-city"]),
    ("% sin precio Río de Janeiro", 8.50, pct_no_price["rio-de-janeiro"]),
    ("% sin precio Santiago", 4.75, pct_no_price["santiago"]),
    ("% sin precio São Paulo", 3.49, pct_no_price["sao-paulo"]),
    ("razón precio / cotización (mín. y máx. = 1)", 1.0, round(price_ratio.min(), 4) if price_ratio.max() == price_ratio.min() else -1),
    ("anuncios con precio y cotización", 182_983, int(both.sum())),
    ("columnas vacías visibles (+ neighborhood_overview = 12)", 11, len(empty_everywhere)),
    ("minimum_nights >= 30", 6_103, int(monthly.sum())),
    ("precios > 20 x la mediana de su ciudad", 692, int((ratio > 20).sum())),
    ("precios < 0,1 x la mediana de su ciudad", 61, int((ratio < 0.1).sum())),
    ("% sin reseñas", 15.5, round(no_reviews.mean() * 100, 1)),
    ("source = 'previous scrape'", 11_701, int(previous.sum())),
]
table = pd.DataFrame(checks, columns=["cifra", "esperado (informe)", "obtenido"])
table["coincide"] = np.where(np.isclose(table["esperado (informe)"], table["obtenido"]), "sí", "NO")
print("Todas coinciden" if (table["coincide"] == "sí").all() else "HAY DIFERENCIAS: corregir reports/01_eda.md")
table

Todas coinciden


,cifra,esperado (informe),obtenido,coincide
0,filas,194628.00,194628.00,sí
1,anfitriones,89964.00,89964.00,sí
2,% anuncios de anfitriones con >= 10,31.00,31.00,sí
3,anuncios sin precio,11642.00,11642.00,sí
4,% sin precio Belice,4.55,4.55,sí
5,% sin precio Bogotá,3.65,3.65,sí
6,% sin precio Buenos Aires,7.12,7.12,sí
7,% sin precio Ciudad de México,6.92,6.92,sí
8,% sin precio Río de Janeiro,8.50,8.50,sí
9,% sin precio Santiago,4.75,4.75,sí
